# Vegetation and delta formation

This notebook builds and inspects the numerical experiment. The domain represents an idealised EM4-sized flume. Vegetation changes hydraulic resistance through the native Baptist model. It does not calculate mechanical root reinforcement. Input plots and engine results are kept separate.

In [ ]:
from pathlib import Path
import json
import sys
from IPython.display import display, Image

ROOT = Path.cwd()
if not (ROOT / "build_cases.py").is_file():
    raise FileNotFoundError("Open the notebook from the project folder")
print("Interpreter:", sys.executable)
config_path = ROOT / "experiment.json"
config = json.loads(config_path.read_text())
config

## Initial conditions

The model has a sloping floodplain, an initial channel and a flooded basin. Discharge is 35 mL/s. Geometry, sediment properties and plant dimensions remain scenario values. The bed contains a finite erodible sediment layer; no external sediment feed is prescribed. The green areas below show the initial vegetation mask, not simulated plant growth or a predicted delta.

Flow is established for 60 seconds on a fixed bed before sediment transport starts. The configured duration is the additional active morphological time. Restarted continuations skip this conditioning.

In [ ]:
from analyse import preview
preview_path = preview(config_path, ROOT / "figures")
display(Image(filename=str(preview_path)))

## Generate the model inputs

The generator writes a UGRID mesh, spatial fields, boundary conditions and sediment/morphology files. All fresh cases use the same bed and forcing. Running this cell generates inputs only; it does not start Delft3D FM. Keep an executed case separate before regenerating its inputs.

In [ ]:
from build_cases import build
CASE_ROOT = ROOT / "runs" / "comparison"
GENERATE_INPUTS = False
if GENERATE_INPUTS:
    if CASE_ROOT.exists():
        raise FileExistsError("Use an empty output folder to preserve previous runs")
    manifest = build(config_path, CASE_ROOT)
    print("Generated", len(manifest["cases"]), "cases")
else:
    print("Input generation is disabled; set GENERATE_INPUTS=True for an empty CASE_ROOT.")

## What the equations calculate

Water continuity is $\partial_t h+\nabla\cdot(h\mathbf u)=s_w$. The depth-averaged momentum balance includes gravity, bed friction and vegetation resistance. An idealised stem force per horizontal area is $\mathbf F_v=\tfrac12\rho C_DNd\min(h,H_v)|\mathbf u|\mathbf u$; the engine uses its Baptist closure.

The Exner balance is $(1-p)\partial_t z_b+\nabla\cdot\mathbf q_b=D-E$. Bed change therefore depends on sediment-flux divergence and exchange with suspended material. Root cohesion is a different mechanism and is not represented by changing drag.

For face area $A_j$, this notebook calculates $V_+=\sum_j A_j\max(\Delta z_j,0)$ and $V_-=\sum_j A_j\max(-\Delta z_j,0)$. These are positive and negative **net bed changes between stored maps**, not gross erosion/deposition over the entire run. Their difference is not a sediment conservation residual: that also needs boundary transport and suspended inventory.

`check_budget.py` tests native suspended and bed storage against integrated boundary transport. The raw water ledger omits moving-bed displacement; the analysis reports that limitation rather than treating a corrected ledger as physical validation.

## Read an executed case

Select the actual native map file after running the project launcher. The reader requires documented FM time-varying bed fields and aligned cell areas. It rejects missing results and unsupported schemas. In particular, the static `mesh2d_flowelem_bl` field cannot substitute for an evolving bed.

In [ ]:
from analyse import analyse
RESULTS_ROOT = ROOT / "results" / "comparison"
CASE_NAME = "bare"
case_dir = CASE_ROOT / CASE_NAME
map_files = sorted((case_dir / "output").glob("*_map.nc"))
if len(map_files) == 1:
    record = json.loads((case_dir / "run_record.json").read_text())
    if record["status"] != "executed_to_requested_time":
        raise RuntimeError("Finish the selected simulation before analysing it")
    metrics = analyse(map_files[0], config_path, RESULTS_ROOT / CASE_NAME,
                      scenario=CASE_NAME, threshold=0.001)
    print(json.dumps(metrics, indent=2))
    display(Image(filename=str(RESULTS_ROOT / CASE_NAME / "bed_change.png")))
    display(Image(filename=str(RESULTS_ROOT / CASE_NAME / "bed_3d.png")))
elif not map_files:
    print("No native map output found. No simulation results have been calculated here.")
else:
    raise RuntimeError("Select a single serial map explicitly; do not combine partitions by filename.")

## Compare matching cases

Use the same elapsed interval, initial bed, discharge and sediment settings. The downstream deposition footprint counts cells beyond the initial basin margin with at least the selected bed-change threshold. It can include submerged deposits; it is not the area of newly emergent land. The downstream centroid weights positive bed change by cell area.

Inspect conservation, numerical resolution and threshold sensitivity before interpreting a difference. A shorter or differently initialised run is not a matched control. The available essay observations permit a qualitative comparison; they do not calibrate the numerical values.

In [ ]:
records = []
for name in config["cases"]:
    metrics_path = RESULTS_ROOT / name / "metrics.json"
    if metrics_path.is_file():
        r = json.loads(metrics_path.read_text())
        records.append({k: r[k] for k in ["scenario", "map_duration_s",
            "positive_bed_change_bulk_volume_m3", "negative_bed_change_bulk_volume_m3",
            "downstream_deposition_footprint_m2", "downstream_deposition_centroid_x_m"]})
if records:
    display(records)
else:
    print("No analysed native results available.")
comparison_path = RESULTS_ROOT / "comparison.png"
if comparison_path.is_file():
    display(Image(filename=str(comparison_path)))


## Inspect the result in ParaView

The native calculation is depth averaged. ParaView displays its evolving bed and water surface in three dimensions and can animate the saved times. The exported geometry retains cell-centred bed levels; vertical exaggeration affects the view only. Velocity has two computed horizontal components.

Open `paraview/<case>/delta.pvd` in ParaView and press **Apply**. The saved `.pvsm` restores the figure settings. `docs/PARAVIEW.md` explains how to export a new run and regenerate the image using ParaView's own Python.

In [ ]:
paraview_dir = ROOT / "paraview" / CASE_NAME
image_path = paraview_dir / "bed_change.png"
if image_path.is_file():
    display(Image(filename=str(image_path)))
    print("ParaView time series:", paraview_dir / "delta.pvd")
else:
    print("No ParaView render yet. Use export_paraview.py and render_paraview.py after completing the model.")